In [ ]:
from cfgnp.train_suite import train_run, ExperimentFactory
from cfgnp.graph_approach.train_graph import eval_test_graph
import torch
from cfgnp.util.util import DEVICE

from cfgnp.util.data_paths import OtherExperimentsPath
from cfgnp.graph_approach.train_graph import create_dataset_artificial_graph
from torch_geometric.loader import DataLoader
from cfgnp.util.util import REGRESSION_TARGET_INDICES, DEVICE
import matplotlib.pyplot as plt

other_sizes = [15]
for size in other_sizes:
    # config = ExperimentFactory.create(f"artificial_{size}", True, 100, True, 1e-4).build()
    model_config = ExperimentFactory.create("artificial_10", True, 100, True, 1e-4).build()
    model = model_config.model

    testset = create_dataset_artificial_graph([], cache_path=OtherExperimentsPath.get_graph_ds_path(f"artificial_{size}", "test"))
    default_cache_path = OtherExperimentsPath.get_graph_ds_path(f"artificial_{size}", "test")
    create_dataset_artificial_graph(testset, cache_path=default_cache_path)
    test_loader = DataLoader(testset, batch_size=4, shuffle=False)

    model.load_state_dict(torch.load("/data/coml-intersection-joins/lina4921/artifacts//graph_model_19-08-2026_02-22.pt"))
    model.to(device=DEVICE)
    test_loss = eval_test_graph(test_loader, model, model_config.loss_fn, None, model_config.target_indice_map, DEVICE, False)
    print(str(size), test_loss)

In [ ]:
from cfgnp.graph_approach.train_graph import eval_test_graph
from cfgnp.train_suite import ExperimentFactory
import torch
from cfgnp.util.util import DEVICE

datasets = ["triangle_LIN", "triangle_NLIN", "triangle_NADD", "mshape_LIN", "mshape_NLIN", "mshape_NADD"]
models = {
    "mshape_LIN": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_15-08-2026_18-35.pt",
    "mshape_NADD": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_15-08-2026_21-19.pt",
    "mshape_NLIN": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_15-08-2026_21-18.pt",
    "triangle_LIN": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_15-08-2026_23-33.pt",
    "triangle_NADD": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_16-08-2026_01-34.pt",
    "triangle_NLIN": "/data/coml-intersection-joins/lina4921/artifacts//graph_model_16-08-2026_01-42.pt"
}
results = {}
for dataset in datasets:
    config = ExperimentFactory.create(dataset, True, 100, True, 1e-4).build()
    results[dataset] = {}
    model = config.model
    model.load_state_dict(torch.load(models[dataset]))
    model.to(device=DEVICE)
    test_loss = eval_test_graph(config.test_loader, model, config.loss_fn, None, config.target_indice_map, DEVICE, False)
    results[dataset] = test_loss

In [ ]:
from cfgnp.train_suite import CheXpertGeneralConfig, CheXpertOODNodeDropConfig
import torch

cfg = CheXpertOODNodeDropConfig(3, True, "chexpert_ood_3", True, size_invariant=True)
model = cfg.model
model.load_state_dict(torch.load("/data/coml-intersection-joins/lina4921/artifacts/graph_model_24-08-2026_22-01.pt"))

# cfg = CheXpertGeneralConfig("chexpert", True)
# model = cfg.model
# model.load_state_dict(torch.load("/data/coml-intersection-joins/lina4921/artifacts//graph_model_18-08-2026_20-09.pt"))

big_config = CheXpertGeneralConfig("chexpert", True)
big_model = big_config.model
model.target_classifiers = big_model.target_classifiers
model.num_features = big_model.num_features
model.max_classes = big_model.max_classes
model.class_counts = big_model.class_counts


In [ ]:
from cfgnp.loss import compute_per_index_metrics_loader
from cfgnp.util.data_paths import ChexpertPath
from cfgnp.graph_approach.train_graph import create_dataset_artificial_graph
from torch_geometric.loader import DataLoader
from cfgnp.util.util import REGRESSION_TARGET_INDICES, DEVICE
import matplotlib.pyplot as plt
import numpy as np
import os

testset = create_dataset_artificial_graph([], cache_path=ChexpertPath.get_graph_ds_path("test"), target_num_nodes=4)

test_loader = DataLoader(testset, batch_size=4, shuffle=False)
model = model.to(device=DEVICE)
model.eval()

generated_images_dir = "/data/coml-intersection-joins/lina4921/results/chexpert_ood3_generated_images"

for intervention_idx, d in enumerate(test_loader):
    orig_image = d.x_orig.reshape((d.batch_size, -1, 3))[:, 4:].reshape((-1, 128, 128, 3))
    # print("Original features ", d.x_orig[:4, 0, 0], ", intervened features ", d.x_int[:4, 0, 0])
    orig_features = d.x_orig[:4, 0, 0]
    int_features = d.x_int[:4, 0, 0]
    dec_orig, _, _ = model.vae.forward(orig_image.to(cfg.device).permute(0, 3, 1, 2))
    dec_img = dec_orig[0].detach().cpu().permute(1, 2, 0).numpy()
    dec_vis = (dec_img - dec_img.min()) / (dec_img.max() - dec_img.min())

    output = model.predict_image(d.to(cfg.device))
    img = output[0].detach().cpu().permute(1, 2, 0).numpy().clip(min=-1, max=1)
    img_vis = (img - img.min()) / (img.max() - img.min())

    # Difference computed on the raw (unnormalized) decoded images so that the two
    # per-image min/max normalizations used for display don't distort the comparison.
    diff = np.abs(img - dec_img).mean(axis=-1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(dec_vis)
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(img_vis)
    axes[1].set_title("Counterfactual")
    axes[1].axis("off")
    heatmap = axes[2].imshow(diff, cmap="hot")
    axes[2].set_title("|Factual - Counterfactual|")
    axes[2].axis("off")
    fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
    fig.tight_layout()
    # plt.show()

    intervention_dir = os.path.join(generated_images_dir, f"counterfactual_{intervention_idx}")
    os.makedirs(intervention_dir, exist_ok=True)
    fig.savefig(os.path.join(intervention_dir, "image.png"))
    plt.close(fig)

    with open(os.path.join(intervention_dir, "features.txt"), "w") as f:
        f.write(f"Original features: {orig_features.tolist()}\n")
        f.write(f"Counterfactual features: {int_features.tolist()}\n")

compute_per_index_metrics_loader(test_loader, model, model.class_counts, REGRESSION_TARGET_INDICES, big_config.target_indice_map, cfg.device)

In [ ]:
valset = create_dataset_artificial_graph([], cache_path=ChexpertPath.get_graph_ds_path("val"), target_num_nodes=4)

val_loader = DataLoader(valset, batch_size=4, shuffle=False)

compute_per_index_metrics_loader(val_loader, model, model.class_counts, REGRESSION_TARGET_INDICES, cfg.target_indice_map, cfg.device)

## Training curves from `runs/` and `outputs_*/`

`outputs_*/` directories hold per-attribute classifier training runs (`training_history.json`
with `train_loss`/`val_loss` plus either `val_mae` for regression targets or
`val_roc_auc`/`val_pr_auc` for classification targets).

`runs/` only contains Lightning `.ckpt` checkpoints from an external diffusion codebase with no
separate metrics log alongside them, so there is nothing to plot from it.

In [ ]:
import json
import re
from pathlib import Path

REPO_ROOT = Path("..").resolve()


def _label_for(history_path: Path, outputs_root: Path) -> str:
    """"outputs_AP/PA_04-07-2026_11-12/training_history.json" -> "AP/PA"."""
    parts = history_path.relative_to(outputs_root).parent.parts
    cleaned = []
    for part in parts:
        part = re.sub(r"^outputs_", "", part)
        part = re.sub(r"_\d{2}-\d{2}-\d{4}_\d{2}-\d{2}$", "", part)
        if part:
            cleaned.append(part)
    return "/".join(cleaned) or history_path.parent.name


history_files = sorted(REPO_ROOT.glob("outputs_*/**/training_history.json"))
histories = {_label_for(p, REPO_ROOT): json.load(open(p)) for p in history_files}
print(f"Found {len(histories)} training runs: {list(histories)}")

runs_dir = REPO_ROOT / "runs"
if runs_dir.exists() and not any(runs_dir.rglob("*.json")):
    print(f"Note: {runs_dir} only has model checkpoints, no logged metrics to plot.")


In [ ]:
import matplotlib.pyplot as plt

TRAIN_COLOR = "#2a78d6"
VAL_COLOR = "#eb6834"
SECONDARY_COLORS = ["#1baf7a", "#eda100"]


def _style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#c3c2b7")
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.grid(True, color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)


n = len(histories)
fig, axes = plt.subplots(n, 2, figsize=(11, 3.5 * n), squeeze=False)

for row, (label, history) in enumerate(histories.items()):
    epochs = range(1, len(history["train_loss"]) + 1)
    ax_loss, ax_metric = axes[row]

    ax_loss.plot(epochs, history["train_loss"], color=TRAIN_COLOR, linewidth=2, label="train loss")
    ax_loss.plot(epochs, history["val_loss"], color=VAL_COLOR, linewidth=2, label="val loss")
    ax_loss.set_title(f"{label}: loss")
    ax_loss.set_xlabel("epoch")
    ax_loss.set_ylabel("loss")
    ax_loss.legend(frameon=False)
    _style_axis(ax_loss)

    if any(v is not None for v in history.get("val_mae", [])):
        ax_metric.plot(epochs, history["val_mae"], color=SECONDARY_COLORS[0], linewidth=2, label="val MAE")
        ax_metric.set_title(f"{label}: val MAE")
        ax_metric.set_ylabel("MAE")
    else:
        if any(v is not None for v in history.get("val_roc_auc", [])):
            ax_metric.plot(epochs, history["val_roc_auc"], color=SECONDARY_COLORS[0], linewidth=2, label="val ROC-AUC")
        if any(v is not None for v in history.get("val_pr_auc", [])):
            ax_metric.plot(epochs, history["val_pr_auc"], color=SECONDARY_COLORS[1], linewidth=2, label="val PR-AUC")
        ax_metric.set_title(f"{label}: val AUC")
        ax_metric.set_ylabel("AUC")
    ax_metric.set_xlabel("epoch")
    ax_metric.legend(frameon=False)
    _style_axis(ax_metric)

fig.tight_layout()
plt.show()
